# 02 — Pilot benchmark (RMAT scale 16)

Run only after Notebook 01 has completed successfully. This pilot uses one synthetic RMAT graph (`2^16` vertices, edge factor 8), MPI ranks 1, 2 and 4, original/random vertex IDs, blocking/non-blocking communication, and 2 BFS sources per configuration: **12 configurations / 24 BFS traversals**.

Open MPI is reinstalled/selected explicitly in a fresh runtime, and the notebook probes communicator sizes 1, 2 and 4 before launching the pilot. Every configuration is checked against its requested rank count in the run log and all three CSV outputs.

**Success ZIP rule:** the pilot ZIP is created/downloaded only after the complete matrix, reports, plots and presentation pass validation. Failed/incomplete runs retain Drive checkpoints but do not create a ZIP.

In [1]:
from pathlib import Path
import os, sys, shutil, json, traceback
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
WORKSPACE = Path('/content/drive/MyDrive/Distributed_BFS_MPI_Colab')
SOURCE_DIR = WORKSPACE / 'Distributed_BFS_MPI'
WORK_DIR = Path('/content/Distributed_BFS_MPI')
RUN_DIR = WORKSPACE / 'runs' / 'pilot'
ARCHIVE_NAME = '02_pilot_benchmark_results.zip'

# Remove any prior archive before checking prerequisites; a failed rerun must not
# leave a stale ZIP that could be mistaken for the current execution.
if (WORKSPACE / 'downloads').exists():
    (WORKSPACE / 'downloads' / ARCHIVE_NAME).unlink(missing_ok=True)
(Path('/content') / ARCHIVE_NAME).unlink(missing_ok=True)
sys.path.insert(0, str(WORKSPACE))
import colab_helpers as cb
cb.clear_previous_archive(WORKSPACE, ARCHIVE_NAME)
if not (SOURCE_DIR / 'src' / 'distributed_bfs.cpp').is_file():
    raise FileNotFoundError(f'Missing source project: {SOURCE_DIR}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'SUCCESS.json').unlink(missing_ok=True)
(RUN_DIR / 'FAILURE.json').unlink(missing_ok=True)
(RUN_DIR / 'RUN_COMPLETE.json').unlink(missing_ok=True)
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(SOURCE_DIR, WORK_DIR)

# Refuse benchmarking unless Notebook 01 proves two actual ranks and the same source.
cb.require_correctness_passed(WORKSPACE, SOURCE_DIR)
print('Notebook 01 two-rank correctness prerequisite: PASS')
print('Drive result folder:', RUN_DIR)
print('Local source copy:', WORK_DIR)

Mounted at /content/drive
Notebook 01 two-rank correctness prerequisite: PASS
Drive result folder: /content/drive/MyDrive/Distributed_BFS_MPI_Colab/runs/pilot
Local source copy: /content/Distributed_BFS_MPI


In [3]:
# Fix the validation error in colab_helpers.py directly on Drive
helper_path = WORKSPACE / 'colab_helpers.py'
if helper_path.exists():
    code = helper_path.read_text(encoding='utf-8')

    # Replace the strict string matching validation to accommodate '.openmpi' / 'OpenRTE'
    old_check = 'if "open mpi" not in compiler_version.lower() or "open mpi" not in launcher_version.lower():'
    new_check = 'if not (any(x in compiler_version.lower() for x in ["open mpi", "openmpi"]) and any(x in launcher_version.lower() for x in ["open mpi", "openmpi", "openrte"])):'

    if old_check in code:
        code = code.replace(old_check, new_check)
        helper_path.write_text(code, encoding='utf-8')
        print("Successfully patched colab_helpers.py validation check!")

        # Reload the modified module
        import importlib
        importlib.reload(cb)
    else:
        print("Target check string not found. It might have been modified already, or we need to review helper_path.")
else:
    print(f"Could not find {helper_path}")

Successfully patched colab_helpers.py validation check!


## Run the pilot and create the success-only ZIP

The run is resumable. If Colab disconnects, reopen the same notebook and rerun it with the same settings; completed configurations are validated and skipped. Partial/invalid configurations are rerun. Do not alter the pilot settings when resuming.

In [4]:
try:
    mpi = cb.install_openmpi_and_build(
        WORK_DIR, WORKSPACE, process_counts=[1, 2, 4],
        install=True, install_python_requirements=True,
    )
    metadata = RUN_DIR / 'metadata'
    metadata.mkdir(parents=True, exist_ok=True)
    (metadata / 'openmpi_toolchain.json').write_text(json.dumps(mpi['report'], indent=2), encoding='utf-8')

    run_settings = {
        'stage': 'pilot',
        'datasets': ['RMAT_s16'],
        'ranks': [1, 2, 4],
        'orderings': ['original', 'random'],
        'modes': ['blocking', 'nonblocking'],
        'repetitions': 2,
        'scales': [16],
        'edge_factor': 8,
        'run_snap': False,
        'run_rmat': True,
        'mpi_compiler': mpi['mpicxx'],
        'mpi_launcher': mpi['mpirun'],
        'mpi_version': mpi['report']['launcher_version'].splitlines()[0],
    }
    cb.ensure_run_config(RUN_DIR, run_settings, SOURCE_DIR)

    env = mpi['env'].copy()
    env.update({
        'OUT_DIR': str(RUN_DIR),
        'RANKS': '1 2 4',
        'REPETITIONS': '2',
        'SCALES': '16',
        'EDGE_FACTOR': '8',
        'RUN_SNAP': '0',
        'RUN_RMAT': '1',
        'RESUME': '1',
        'MPIEXEC': mpi['mpirun'],
        'MPICXX': mpi['mpicxx'],
        'MPIEXEC_FLAGS': '--allow-run-as-root --oversubscribe --bind-to none',
        'RUNNER_VERSION': cb.RUNNER_VERSION,
        'SOURCE_FINGERPRINT': cb.source_fingerprint(SOURCE_DIR),
    })
    cb.run_stream(
        ['bash', str(WORK_DIR / 'scripts' / 'run_experiments.sh')],
        cwd=WORK_DIR, env=env, log_path=RUN_DIR / 'colab_runner.log',
        timeout=None, check=True,
    )

    completion = cb.validate_complete_run(
        RUN_DIR, datasets=['RMAT_s16'], ranks=[1, 2, 4],
        orderings=['original', 'random'], modes=['blocking', 'nonblocking'],
        repetitions=2,
    )
    if completion['configurations'] != 12 or completion['bfs_summary_rows'] != 24:
        raise RuntimeError(f'Pilot matrix size mismatch: {completion}')

    cb.write_success_manifest(RUN_DIR, '02', {
        'source_fingerprint': cb.source_fingerprint(SOURCE_DIR),
        'completion': completion,
        'mpi_implementation': 'Open MPI',
    })
    cb.create_success_zip(RUN_DIR, WORKSPACE, ARCHIVE_NAME)
    print('\nPILOT COMPLETE: 12 configurations, 24 BFS traversals, all validations passed.')
    print('Automatically downloaded:', ARCHIVE_NAME)

except BaseException as exc:
    (RUN_DIR / 'SUCCESS.json').unlink(missing_ok=True)
    (RUN_DIR / 'FAILURE.json').write_text(json.dumps({
        'success': False, 'notebook': '02',
        'error_type': type(exc).__name__, 'error': str(exc),
        'traceback': traceback.format_exc(),
        'failed_at_utc': datetime.now(timezone.utc).isoformat(),
    }, indent=2), encoding='utf-8')
    print('\nFAILED/INCOMPLETE: no results ZIP was created.')
    print('Validated checkpoints and diagnostics, if any, remain in:', RUN_DIR)
    raise


$ /usr/bin/mpicxx.openmpi -O2 /content/mpi_two_rank_probe.cpp -o /content/mpi_two_rank_probe
Verified Open MPI -np 1: ranks [0], communicator size 1.
Verified Open MPI -np 2: ranks [0, 1], communicator size 2.
Verified Open MPI -np 4: ranks [0, 1, 2, 3], communicator size 4.

$ python3 -m pip install -q -r /content/Distributed_BFS_MPI/requirements.txt

$ bash /content/Distributed_BFS_MPI/scripts/build.sh
In file included from /usr/lib/x86_64-linux-gnu/openmpi/include/openmpi/ompi/mpi/cxx/mpicxx.h:277,
                 from /usr/lib/x86_64-linux-gnu/openmpi/include/mpi.h:2911,
                 from /content/Distributed_BFS_MPI/src/distributed_bfs.cpp:1:
/usr/lib/x86_64-linux-gnu/openmpi/include/openmpi/ompi/mpi/cxx/op_inln.h: In member function ‘virtual void MPI::Op::Init(void (*)(const void*, void*, int, const MPI::Datatype&), bool)’:
/usr/lib/x86_64-linux-gnu/openmpi/include/openmpi/ompi/mpi/cxx/op_inln.h:121:25: warning: cast between incompatible function types from ‘void (*)(void*,

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PILOT COMPLETE: 12 configurations, 24 BFS traversals, all validations passed.
Automatically downloaded: 02_pilot_benchmark_results.zip


## Pilot output

Successful execution automatically downloads `02_pilot_benchmark_results.zip` and also saves it under `MyDrive/Distributed_BFS_MPI_Colab/downloads/`. Send that ZIP here for review before Notebook 03.

If the runtime disconnects, no success ZIP is produced. Re-run this notebook with the same settings to resume from validated configuration checkpoints.